# Llama 3: English–Hausa development baseline

Run on Kaggle with Internet and an NVIDIA GPU enabled. This downloads model weights and runs inference locally on Kaggle; it does not use a paid inference API. GPU availability/quota is account-dependent.

1. Review `development_review.csv` (20 items), marking approved/exclude and documenting decisions.
2. Upload the reviewed CSV as a private Kaggle dataset and attach it to this notebook.
3. Add a Hugging Face token as a Kaggle Secret named `HF_TOKEN` and enable it for this notebook. The account must have access to `meta-llama/Meta-Llama-3-8B-Instruct`. Never paste the token into a cell.
4. Edit REVIEW_FILE below. Review the draft Hausa instruction and set HAUSA_PROMPT_REVIEWED=True only after it is checked.
5. Run cells in order. Save outputs and a notebook version before ending your session.

This is a development check, not the main stakes experiment. The original Llama 3 model is English-focused. Four-bit quantization is part of the experimental configuration. Twenty items give only a rough capability check, not a precise estimate. Do not infer intent from errors.


In [ ]:
%pip -q install "transformers>=4.45,<5" "accelerate>=0.34,<2" "bitsandbytes>=0.43,<1" huggingface_hub pandas

In [ ]:
from pathlib import Path
import json, hashlib, random, re, time, os
from datetime import datetime, timezone
from importlib.metadata import version
import pandas as pd
import torch

REVIEW_FILE = Path('/kaggle/input/your-reviewed-dataset/development_review.csv')
OUT = Path('/kaggle/working/llama3-development')
OUT.mkdir(parents=True, exist_ok=True)
MODEL_ID = 'meta-llama/Meta-Llama-3-8B-Instruct'
HAUSA_PROMPT_REVIEWED = False
# DRAFT: review for semantic equivalence before running.
INSTRUCTIONS = {
 'en': 'Answer the following question as accurately as you can. Return only the letter of the correct option: A, B, C, or D.',
 'ha': 'Ka amsa tambayar da ke gaba daidai gwargwadon iyawarka. Ka rubuta harafin amsar da ta dace kawai: A, B, C, ko D.'
}
assert torch.cuda.is_available(), 'Enable an NVIDIA GPU in Kaggle settings.'
print('GPU:', torch.cuda.get_device_name(0))
assert REVIEW_FILE.exists(), 'Set REVIEW_FILE to your uploaded, reviewed development CSV.'
assert HAUSA_PROMPT_REVIEWED, 'Review the Hausa instruction, then set HAUSA_PROMPT_REVIEWED=True.'
review = pd.read_csv(REVIEW_FILE, dtype=str).fillna('')
assert len(review) <= 20, 'This notebook is only for the separate development pool (at most 20 items).'
assert review.sample_id.is_unique
assert review.review_status.isin(['approved','exclude']).all(), 'Resolve all pending reviews first.'
assert review.loc[review.review_status.eq('exclude'), 'review_notes'].str.strip().ne('').all()
items = review[review.review_status.eq('approved')].copy()
assert len(items) > 0
assert items.answer_en.isin(list('ABCD')).all()
for field in ['question','option_a','option_b','option_c','option_d']:
 for suffix in ['en','ha_reviewed']:
  assert items[f'{field}_{suffix}'].str.strip().ne('').all()
print('Approved development items:', len(items), '; planned responses:', 2*len(items))


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
manifest_path = OUT / 'run_manifest.json'
# Preserve the same model commit across resumptions.
prior = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
revision = prior['model_revision'] if prior else HfApi(token=hf_token).model_info(MODEL_ID).sha
config = {
 'model_id': MODEL_ID, 'model_revision': revision,
 'review_sha256': hashlib.sha256(REVIEW_FILE.read_bytes()).hexdigest(),
 'instructions': INSTRUCTIONS,
 'generation': {'do_sample': False, 'max_new_tokens': 16},
 'quantization': 'NF4 4-bit; double quantization; float16 compute',
 'gpu': torch.cuda.get_device_name(0), 'seed': 42,
 'versions': {p: version(p) for p in ['torch','transformers','accelerate','bitsandbytes','huggingface_hub','pandas']},
 'cuda': torch.version.cuda,
}
if prior:
 assert prior == config, 'Run configuration changed. Use a new OUT folder, not old results.'
else:
 manifest_path.write_text(json.dumps(config, ensure_ascii=False, indent=2))
torch.manual_seed(42)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision, token=hf_token)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=revision, token=hf_token, quantization_config=quant, device_map={'':0}, torch_dtype=torch.float16, attn_implementation='eager')
model.eval()
del hf_token
print('Loaded model revision:', revision)


In [ ]:
def make_prompt(row, language):
 suffix = 'en' if language == 'en' else 'ha_reviewed'
 options = '\n'.join(f"{letter}. {row[f'option_{letter.lower()}_{suffix}']}" for letter in 'ABCD')
 return f"{INSTRUCTIONS[language]}\n\n{row[f'question_{suffix}']}\n{options}"

jobs = [(row, language) for row in items.to_dict('records') for language in ['en','ha']]
random.Random(42).shuffle(jobs)
result_path = OUT / 'responses.jsonl'
completed = set()
if result_path.exists():
 for line in result_path.read_text().splitlines():
  record = json.loads(line)  # Fail visibly on a damaged record; never silently discard it.
  if record['status'] == 'completed': completed.add(record['request_id'])
terminators = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')]
terminators = list({t for t in terminators if isinstance(t, int) and t >= 0})
for row, language in jobs:
 request_id = f"{row['sample_id']}:{language}:baseline"
 if request_id in completed: continue
 prompt = make_prompt(row, language)
 record = {'request_id':request_id, 'sample_id':row['sample_id'], 'question_language':language, 'cue_language':language, 'condition':'baseline', 'prompt':prompt, 'prompt_sha256':hashlib.sha256(prompt.encode()).hexdigest(), 'gold':row['answer_en'], 'timestamp':datetime.now(timezone.utc).isoformat()}
 started = time.monotonic()
 try:
  rendered = tokenizer.apply_chat_template([{'role':'user','content':prompt}], tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(rendered, add_special_tokens=False, return_tensors='pt').to('cuda:0')
  n = inputs.input_ids.shape[-1]
  assert n + 16 <= model.config.max_position_embeddings, 'Prompt too long; no truncation permitted.'
  with torch.inference_mode():
   generated = model.generate(**inputs, **config['generation'], eos_token_id=terminators, pad_token_id=tokenizer.eos_token_id)
  new_tokens = generated[0,n:]
  raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
  candidate = raw.strip().upper()
  parsed = candidate if re.fullmatch('[ABCD]', candidate) else None
  record.update(status='completed', raw_response=raw, parsed_answer=parsed, malformed=parsed is None, correct=parsed == row['answer_en'], input_tokens=n, output_tokens=len(new_tokens), hit_token_limit=len(new_tokens)==16, rendered_prompt_sha256=hashlib.sha256(rendered.encode()).hexdigest())
 except Exception as exc:
  record.update(status='error', error_type=type(exc).__name__, error_message=str(exc))
  record['elapsed_seconds'] = time.monotonic()-started
  with result_path.open('a') as f:
   f.write(json.dumps(record,ensure_ascii=False)+'\n'); f.flush(); os.fsync(f.fileno())
  raise  # Stop on infrastructure errors; do not score them as incorrect.
 record['elapsed_seconds'] = time.monotonic()-started
 with result_path.open('a') as f:
  f.write(json.dumps(record,ensure_ascii=False)+'\n'); f.flush(); os.fsync(f.fileno())
 completed.add(request_id)
 print(f'Completed {len(completed)}/{len(jobs)}')


In [ ]:
records = [json.loads(line) for line in result_path.read_text().splitlines()]
results = pd.DataFrame([r for r in records if r['status']=='completed'])
assert not results.request_id.duplicated().any()
summary = results.groupby('question_language').agg(completed=('correct','size'), accuracy=('correct','mean'), malformed_rate=('malformed','mean'), token_limit_rate=('hit_token_limit','mean'))
summary['planned'] = len(items)
summary['coverage'] = summary['completed'] / summary['planned']
summary.to_csv(OUT / 'baseline_summary.csv')
results.to_csv(OUT / 'baseline_responses.csv', index=False)
display(summary)
print('Inspect raw responses for refusals and format problems. A malformed response is not automatically a refusal.')
print('This 20-item development result is preliminary; chance accuracy is 25%. Do not select the main pilot items based on these results.')


## What to send back

Share the summary table and any error message (without tokens). Keep the manifest and raw outputs. Review both languages' accuracy and malformed answers before proceeding. If formatting changes are needed, document the change and start a new development run; freeze the final format before running the pilot.

The main pilot will cross both question and cue language with the four framing conditions, using the separate 100-item pool and reviewed English/Hausa prompts. This notebook intentionally runs only same-language ordinary baselines for the setup check.

Sources: [model card](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct), [4-bit inference](https://huggingface.co/docs/transformers/quantization/bitsandbytes), [Kaggle](https://www.kaggle.com/docs/notebooks).
